# Journal de bord

## 📅 Journal de bord – Jour 1 (18/09/2026)

### 🔧 Étapes / Actions réalisées
- Collections des data
- Organisation du squelette
- Préparation du git

### 📝 Observations / Difficultés
- /

### ✅ Prochaines étapes
- Exploration des données

## 📅 Journal de bord – Jour 2 (27/09/2026)


### 🔧 Étapes / Actions réalisées
- Relecture intégrale du sujet - entretien client, cartographie sources, risques RGPD croisés, datasheet Gebru, grille de décision, traçage d'expériences
- Ajout de 5 compléments ciblés au §1 du notebook : cartographie de la source unique, mécanisme de ré-identification par croisement (`code_insee_commune`), condition de bascule TF-IDF → embeddings, annonce du futur format de traçage


### 📝 Observations / Difficultés
- Notebook resté l'unique livrable (pas de fichier `identification_sources.md` séparé)


### ✅ Prochaines étapes
- Étape 2 : Exploration


## 📅 Journal de bord – Jour 3 (28/09/2026)


### 🔧 Étapes / Actions réalisées
- Mise en place de l'environnement d'exécution
- Audit qualité complet du CSV : manquants confirmés, 74 lignes incohérentes confirmées, code ROME `W1401` identifié comme non conforme à la nomenclature officielle (lettre absente de A-N)
- Vérification PII sur `synthese_entretien` : aucune trouvée (9 phrases-gabarits génériques)
- Calcul du DI sur `nationalite_hors_ue`
- Création de `src/trajectoire_emploi/fairness.py` 
- Brouillon de datasheet Gebru et dictionnaire de variables ajoutés au notebook
- Notebook vérifié exécutable de bout en bout (`nbconvert --execute`, 0 erreur)


### 📝 Observations / Difficultés
- Aucun package Python n'était installé sur la machine au départ 
- `.gitignore` à compléter pour `.venv/`


### ✅ Prochaines étapes
- Étape 3 : Qualité et biais (traitement des 74 lignes incohérentes, code W1401, bilan éthique n°2)


## 📅 Journal de bord – Jour 4 (29/09/2026)


### 🔧 Étapes / Actions réalisées
- Étape 3 réalisée : crosstab cible × nationalité sur 3 classes 
- Rédaction bilan éthique n°2
- Ajout de 4 visualisations EDA — distribution de la cible, distribution de l'âge, boxplot de l'ancienneté (données brutes), heatmap cible × nationalité  

- Étape 4 (Préparation) réalisée : `src/trajectoire_emploi/features.py` (extraction du département depuis le code INSEE, besoin hors ontologie) et `src/trajectoire_emploi/pipeline.py` (4 `ColumnTransformer`, un par scénario) créés, 
- Création de 11 nouveaux tests (`test_features.py`, `test_pipeline.py`)
- Application du split stratifié 2000/500 , pas de split temporel (pas de date dans le jeu de données)

### 📝 Observations / Difficultés
- Bug de chemin `DATA_PATH`/`sys.path` (`../../` vs `../`) non résolu
- Le split n'est pas encore scellé formellement (pas de `outputs/split_scelle.json`)


### ✅ Prochaines étapes
- Étape 5 : Benchmark multi-modèles


## 📅 Journal de bord – Jour 5 (30/09/2026)


### 🔧 Étapes / Actions réalisées
- Étape 5 (Benchmark) réalisée : `src/trajectoire_emploi/evaluation.py` (F1 macro, recall classe 2, kappa pondéré, taux d'erreur 2→0, coût moyen) et `src/trajectoire_emploi/benchmark.py` (boucle scénarios × modèles, mêmes folds)
- Création de 12 nouveaux tests
- Benchmark complet : 4 scénarios × 6 modèles (baseline, régression logistique, RandomForest ×2, LightGBM, XGBoost), `RepeatedStratifiedKFold` (5 folds × 3 répétitions) — meilleur compromis provisoire : LightGBM/XGBoost sur S1 (F1 macro ≈ 0,695)
- Découverte marquante : sur S3 (texte seul), tous les modèles convergent vers un score strictement identique — vérifié : seulement 10 vecteurs TF-IDF distincts (9 phrases-gabarits + 1 pour les manquants)
- Test de robustesse : extrapolation silencieuse détectée sur un âge hors plage d'entraînement (150 ans) ; garde-fou proposé (bornes d'entrée + abstention) pour la future API
- Explication de l'intérêt de la matrice de coûts ajoutée au README

- Étape 6 (Arbitrage) réalisée : `src/trajectoire_emploi/decision.py` (décision à coût minimal) et `src/trajectoire_emploi/calibration.py` (ECE, reliability diagram)
- Création de 6 nouveaux tests
- Vérification de la calibration (ECE ≈ 0,04-0,05) pour les 3 candidats retenus (S1-LightGBM, S1-XGBoost, S2-LightGBM)
- Décision à coût minimal (vs argmax) : divise par plus de deux le taux d'erreur critique 2→0 pour un coût modeste en F1 macro
- Audit d'équité refait sur les **prédictions** (pas l'étiquette) — le modèle **amplifie** le biais sur S1, mais **repasse au-dessus du seuil d'alerte 4/5** sur S2 (DI = 0,838) en retirant seulement `nationalite_hors_ue`
- Choix final tranché : **S2-LightGBM avec décision à coût minimal** — compromis performance/équité argumenté et chiffré

- Étape 7 (Communication) réalisée : `decision.py` enrichi de `cout_attendu_minimal()` et `decision_avec_abstention()` (5 tests) ; analyse d'erreurs (bloc principal = confusion 0↔1, pas 2→0) ; seuil d'abstention retenu à 0,7 (~19 % de revue humaine, aligné sur < 20 %) 
- Rédaction de la note de recommandation client
- **Verdict final sur le test scellé** : F1 macro = 0,722, taux d'erreur critique 2→0 = 0,027, DI sur les prédictions = 0,808 (toujours au-dessus du seuil d'alerte 4/5)
- Traçage du scellement dans `outputs/split_scelle.json` (modèle, seuil, métriques, date d'exécution)

- Étape 8 (FastAPI) réalisée
- Création de `persistence.py` (`construire_metadata`, `persister_modele`, `charger_modele`) — packaging du pipeline complet (préprocesseur + classifieur), 5 clés obligatoires dans le `.json` adjacent (4 tests)
- **Réentraînement final** du pipeline S2-LightGBM sur **train+test combinés** (2500 lignes)
- Packaging : `models/trajectoire_emploi_v1.joblib` + `.json`
- API FastAPI créée (`app/main.py`, `app/schemas.py`, `app/middleware.py`) : `/health`, `/info`, `/predict` avec décision à coût minimal + abstention (seuil 0,7) ; logs structurés avec `request_id`, sans PII
- 10 tests API + contract test du modèle ajoutés — 55 tests passent au total

- Remise en cause de l'argument d'équité : le DI utilisé partout mesurait « prédit classe 0 » comme issue favorable, alors que §3.6 définit le préjudice comme la privation d'accompagnement (classe 2 non détectée) — convention inversée par rapport au préjudice réel
- Refait entièrement : (1a) comparaison taux réel vs prédit de classe 0 par groupe → révèle que S2 prédit « retour rapide » pour le groupe hors-UE **3× plus souvent que la réalité** (31,8 % prédit vs 13,9 % réel) — le DI remonte par perte de pouvoir discriminant, pas par correction ; (1b) recall classe 2 et taux 2→0 **par groupe**, sous la vraie règle : **renversement complet** — S1 protège mieux le groupe hors-UE (recall 0,804, taux 2→0 0,011) que le groupe majoritaire ; S2 inverse ce résultat (recall 0,620, taux 2→0 **0,076, 7× pire qu'avec S1**)
- DI recalculé sous la bonne règle : 0,722 (pas 0,838) pour la convention classe 0, 0,825 pour la convention classe 2 (accompagnement) — les deux nuances absentes du chiffre original
- Création de `experiments.md` (racine, 5 runs documentés de la baseline au modèle de production)

### 📝 Observations / Difficultés
- Bug de chemin `DATA_PATH`/`sys.path` (`../../` vs `../`) non résolu

### ✅ Prochaines étapes
- Étape 9 (Docker) → Étape 10 (CI/CD)


## 📅 Journal de bord – Jour 6 (01/10/2026)


### 🔧 Étapes / Actions réalisées
- Étapes 9-10 réalisées : `services/backend/Dockerfile` (python:3.11-slim, user non-root, healthcheck) + `docker-compose.yml`
- Modèle packagé (`models/trajectoire_emploi_v1.joblib`, 404 Ko) **versionné dans Git** (exception ciblée au `.gitignore`)
- Image testée de bout en bout : 775 Mo, `docker exec whoami` → `appuser`, `docker compose ps` → `healthy`, `/health` `/info` `/predict` (200 et 422) vérifiés sur le vrai conteneur
- `.github/workflows/ci.yml` créé : job `test` (pytest) → job `build` (`needs: test`, build Docker + vérification `/health`) — portée limitée au gate test→build, pas de push GHCR (pas de remote GitHub actif sur ce dépôt)

- Étape 11 réalisée (MLflow) : `scripts/log_experiments_mlflow.py` créé, relogue les 5 runs déjà documentés dans `experiments.md` vers MLflow (params + métriques + tag verdict)
- `mlflow` ajouté à `requirements.txt` (dev uniquement), `mlruns/` ajouté au `.gitignore`
- Vérifié réellement : serveur `mlflow ui` lancé en tâche de fond et interrogé via son API HTTP (`/api/2.0/mlflow/experiments/search`) — 2 expériences retournées (`trajectoire_emploi` + `Default`), 5 runs comparables confirmés

- Étape 13.1 réalisée (Prometheus) : `prometheus-fastapi-instrumentator` sur `services/backend/app/main.py` — `/metrics` ; `Counter` métier `trajectoire_emploi_decisions_total` labellé par décision (4 valeurs, pas de `request_id` en label) ; 
- Création de 3 tests
- Vérifié réellement dans le conteneur Docker : rebuild de l'image, `/predict` puis `/metrics` interrogés sur le conteneur lancé — compteur incrémenté, healthcheck toujours `healthy`

- Étape 13.2 réalisée (Prometheus + Grafana) : `observability/prometheus/prometheus.yml` (scrape `backend:8000`), `observability/grafana/provisioning/` (datasource + provider dashboards) ; `docker-compose.yml` étendu à 3 services avec `depends_on: condition: service_healthy` en cascade
- Stack complète testée réellement : les 3 services démarrent et passent `healthy`, Prometheus scrape le backend (`up{job="backend"}=1` confirmé via son API), Grafana interroge Prometheus via la datasource provisionnée 
- Test de la panne muette : `docker compose stop backend` → `up` bascule à `0` en ~15s — confirme que c'est le seul signal qui détecte vraiment une panne

- Étape 13.3 réalisée (dashboard Grafana) : `trajectoire_emploi_prod.json` — 3 panels (Vie/Vitesse/Comportement, fiche 514)
- Vérifié réellement : dashboard présent dans `GET /api/search` sans import manuel ; trafic généré sur l'API (10 requêtes), les 3 panels interrogés via l'API Grafana renvoient de vraies séries (latence p95 ≈ 0,095s par route, taux de décisions ≈ 0,075/s) — aucun panel en `No data`

- Étape 13.4 réalisée (dérive) : `src/trajectoire_emploi/drift.py` (psi, calculer_ks, calculer_chi2, diagnostic_data_vs_concept_drift) 
- Création de 11 tests
- Démonstration PSI/KS/Chi² train vs test ajouté dans le notebook (proxy, faute de trafic de production réel — limite assumée explicitement dans le texte) + diagnostic data vs concept drift (AUC OOF train 0,838 vs AUC test scellé 0,856, aucun nouveau fit) — aucun signal de dérive détecté, cohérent avec un split stratifié propre

- Étape 13.5 réalisée (runbook + évaluation continue) : `data/reference_set.csv` généré (= ancien X_test, colonnes BRUTES uniquement)
- `scripts/evaluate_model.py` créé : seuils bootstrap (500 ré-échantillonnages, sigma f1_macro ≈ 0,012), golden run gelé, mode `--degrade` pour le chemin rouge
- Le modèle de production a déjà vu `reference_set.csv` pendant son entraînement (réentraîné sur train+test combinés) — golden run artificiellement haut (F1 macro 0,94 au lieu de 0,72 en vrai holdout). Ce script détecte des **régressions de code**, pas la performance réelle
- Chemin vert testé : écart nul, `exit 0`. 
- Chemin rouge testé : `--degrade` → 4 violations détectées, `exit 1`
- `RUNBOOK.md` créé (4 procédures) ; job CI `evaluate-model` branché (`needs: test`) ; 6 tests unitaires sur `verifier_seuils` (75 tests au total)

- Tolérance recalculée à partir de l'écart-type inter-folds de l'Étape 5 = 0,043 → nouvelle tolérance 2×0,043 = 0,086 (arrondi 0,09), remplace l'ancien 0,05 — proxy assumée et documentée dans le script (règle de décision légèrement différente, argmax vs coût minimal+abstention)
- Chemin vert et chemin rouge (`--degrade`) revalidés après correction : toujours 0 violation / 4 violations respectivement — 75 tests pytest toujours verts

- Test complémentaire de dérive (demandé par l'utilisateur) : `scripts/generer_trafic_test.py` + `scripts/comparer_derive_production.py` créés — envoient un trafic synthétique (pas organique) à l'API réellement déployée, comparent à `data/reference_set.csv`
- Testé réellement (stack Docker démarrée, 80 requêtes envoyées) : signal fort détecté sur `anciennete_poste_ans` (PSI 1,2-2,4) et `code_rome_vise` (Chi² p≈0) — identifiés comme des artefacts du générateur (bornes uniformes vs vraie distribution asymétrique ; 5 codes ROME fixes vs dizaines dans la référence), pas une vraie dérive métier — documenté honnêtement dans README et notebook §9.5

### 📝 Observations / Difficultés
- **Bug réel détecté en testant le conteneur** : LightGBM nécessite `libgomp1` (OpenMP), absent de l'image `python:3.11-slim` par défaut — `OSError: libgomp.so.1: cannot open shared object file` au chargement du modèle ; corrigé par `apt-get install libgomp1` dans le Dockerfile
-  Le modèle de production a déjà vu `reference_set.csv` pendant son entraînement (réentraîné sur train+test combinés) — golden run artificiellement haut (F1 macro 0,94 au lieu de 0,72 en vrai holdout). Ce script détecte des **régressions de code**, pas la performance réelle


### ✅ Prochaines étapes
- Étape 12 — interface conseiller + feedback, Étape 14 — réentraînement gardé
